## Data Cleaning and Preparation
What does this dataset do 
- this notebook cleans the data and creates a csv file that will be ready for: 

a. NASA-TLX data analysis 
- This will be the document analyzing the perceived workload of tasks done. 
    
b. Simulation data analysis
- This will be the document analyzing the statistics and the actions done by the participants during the simulation. 

Important note: 
csv files for nasa-tlx should be in 1 folder and csv files for simulation should be in a separate folder

#### Data Description


### Simulation 
| Column Name        | Description                                                  |
|--------------------|--------------------------------------------------------------|
| **timestamp** | recorded time when an occurence occured |
| **phase** | classification of drive |
| **scenario** | Type of occurence that happened|
| **speed_kmh** | recorded speed the participant was going |
| **event** | the violation or occurence type |
| **details** | Split into Violation and Location |
| **Violation** | Specific Violation acquired (?) | 
| **Location** | Exact coordinates the violation occured. This differs every time. |

### NASA TLX 
| Column Name        | Description                                                  |
|--------------------|--------------------------------------------------------------|
| **Participant**   | Participant number for tracking, acts as a unique identifier of a participant |
| **Task Name** | Type of alerts that the participant took (Baseline, AO - Audio Only, HO - Haptic Only, AH - Audio and Haptic) |
| **Mental Demand**| Mental and perceptual activity |
| **Physical Demand**| Physical activity required | 
| **Temporal Demand** | Time pressure according to the pace |
| **Performance** | Successfulness in completing the task |
| **Effort** | 
| **Frustration**| |


#### Data Collection & Methodology 
**NASATLX**
This dataset came from the NASATLX site. The participants were asked to answer the same exact questions. This was done every after drive to gauge their experience immediately after the drive. Upon completing the whole 4 drives, the csv file was generated. 

**Simulation**
This dataset came from the log tracker of the CARLA application, which records the corresponding type of drive and the violations committed. When a violation was done, the coordinates, 


#### Biases and Limitations 

#### Sources and Credits
NASA-TLX site: https://nasa-tlx-calculator.vercel.app/

**Reminders**
- make sure there are 2 separate folders > 1 each for the respective data (1 csv folder for Sim, 1 csv folder for NASATLX)

### Preprocessing

In [95]:
import pandas as pd
import numpy as np
import glob
import re
import json
 

In [96]:
files = glob.glob("/Users/karencafe/Desktop/ThesisTest/SIMCSV/*.csv")

df_simulation = pd.concat(
    (pd.read_csv(f) for f in files),
    ignore_index=True
)

display(df_simulation)


,timestamp,phase,scenario,speed_kmh,event,details
0,1.771922e+09,4,START,0.00,PHASE_START,Assisted-AH
1,1.771923e+09,4,START,0.00,PHASE_START,Assisted-AH
2,1.771923e+09,4,WEBSOCKET,0.00,WS_SEND,"{""phase"": 4, ""speed"": 0.0, ""speed_limit"": 30, ..."
3,1.771923e+09,4,WEBSOCKET,0.00,WS_SEND,"{""phase"": 4, ""speed"": 0.0, ""speed_limit"": 30, ..."
4,1.771923e+09,4,WEBSOCKET,0.00,WS_SEND,"{""phase"": 4, ""speed"": 0.0, ""speed_limit"": 30, ..."
...,...,...,...,...,...,...
748694,1.771682e+09,3,WEBSOCKET,0.00,WS_SEND,"{""phase"": 3, ""speed"": 0.0, ""speed_limit"": 30, ..."
748695,1.771682e+09,3,WEBSOCKET,0.00,WS_SEND,"{""phase"": 3, ""speed"": 0.0, ""speed_limit"": 30, ..."
748696,1.771682e+09,3,WEBSOCKET,0.00,WS_SEND,"{""phase"": 3, ""speed"": 0.0, ""speed_limit"": 30, ..."
748697,1.771682e+09,3,STOP,674.87,PHASE_STOP,Assisted-H


In [97]:
df_simulation["timestamp"] = pd.to_numeric(df_simulation["timestamp"], errors="coerce")
df_simulation["speed_kmh"] = pd.to_numeric(df_simulation["speed_kmh"], errors="coerce")

In [99]:
# removing HUD alerts in phase 1 
df_simulation = df_simulation[
    ~(
        (df_simulation["event"] == "HUD_CUE") &
        (df_simulation["details"].str.strip() == "|")
    )
]

In [100]:
# removing speed with 0-0.1 -> too many and insignificant 
df_simulation = df_simulation[df_simulation["speed_kmh"] > 0.1]

In [101]:
df_simulation

,timestamp,phase,scenario,speed_kmh,event,details
1300,1.771924e+09,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,..."
1301,1.771924e+09,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,..."
1302,1.771924e+09,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,..."
1303,1.771924e+09,4,WEBSOCKET,1.17,WS_SEND,"{""phase"": 4, ""speed"": 1.17, ""speed_limit"": 30,..."
1304,1.771924e+09,4,WEBSOCKET,1.19,WS_SEND,"{""phase"": 4, ""speed"": 1.19, ""speed_limit"": 30,..."
...,...,...,...,...,...,...
748642,1.771682e+09,3,WEBSOCKET,3.53,WS_SEND,"{""phase"": 3, ""speed"": 3.53, ""speed_limit"": 30,..."
748643,1.771682e+09,3,WEBSOCKET,3.53,WS_SEND,"{""phase"": 3, ""speed"": 3.53, ""speed_limit"": 30,..."
748644,1.771682e+09,3,WEBSOCKET,3.41,WS_SEND,"{""phase"": 3, ""speed"": 3.41, ""speed_limit"": 30,..."
748697,1.771682e+09,3,STOP,674.87,PHASE_STOP,Assisted-H


In [102]:
def extract_speed_limit(text):
    if pd.isna(text):
        return None
    
    text = str(text)
    
    # Case 1: JSON-like
    if text.strip().startswith("{"):
        try:
            data = json.loads(text)
            return float(data.get("speed_limit"))
        except:
            return None
    
    # Case 2: Plain text
    match = re.search(r"limit\s([\d.]+)", text)
    if match:
        return float(match.group(1))
    
    return None

df_simulation["speed_limit"] = df_simulation["details"].apply(extract_speed_limit)

In [103]:
display(df_simulation[df_simulation["phase"] == 1].tail(5))

,timestamp,phase,scenario,speed_kmh,event,details,speed_limit
735622,1.771680e+09,1,SPEED_LIMIT,36.00,REACTION,action=VIOLATION_RESOLVED|time=1.78s|reaction_...,NaN
735623,1.771680e+09,1,SPEED_LIMIT,37.01,SPEED_VIOLATION,Speed 37.01 km/h > limit 30.00 km/h at Locatio...,30.0
735624,1.771680e+09,1,SPEED_LIMIT,36.70,REACTION,action=VIOLATION_RESOLVED|time=0.15s|reaction_...,NaN
735625,1.771680e+09,1,STOP,475.51,PHASE_STOP,Baseline,NaN
735626,1.771680e+09,1,STOP,475.51,PHASE_STOP,Baseline,NaN


In [104]:
display(df_simulation['speed_limit'].unique())

array([30., nan, 90., 60.])

In [105]:
# Mark all = 0; start violation = 1; end violation = -1
df_simulation["violation_flag"] = 0
df_simulation.loc[df_simulation["event"] == "SPEED_VIOLATION", "violation_flag"] = 1
df_simulation.loc[df_simulation["details"].str.contains("VIOLATION_RESOLVED", na=False), "violation_flag"] = -1

In [106]:
df_simulation.columns

Index(['timestamp', 'phase', 'scenario', 'speed_kmh', 'event', 'details',
       'speed_limit', 'violation_flag'],
      dtype='object')

In [128]:
display(df_simulation[df_simulation["event"].isin(["SPEED_VIOLATION","REACTION"])].head(30))

,timestamp,phase,scenario,speed_kmh,event,details,speed_limit,violation_flag,event_start,event_id
2591,1.771924e+09,4,SPEED_LIMIT,41.58,REACTION,action=REDUCE_THROTTLE|time=0.76s|control={'th...,NaN,0,0,0
2603,1.771924e+09,4,SPEED_LIMIT,41.77,SPEED_VIOLATION,Speed 41.77 km/h > limit 30.00 km/h at Locatio...,30.0,1,1,1
2604,1.771924e+09,4,SPEED_LIMIT,36.30,REACTION,action=VIOLATION_RESOLVED|time=1.33s|reaction_...,NaN,-1,0,0
4516,1.771924e+09,4,TRAFFIC_LIGHT,2.22,REACTION,action=NO_REACTION|time=3.01s|control={'thrott...,NaN,0,0,0
7549,1.771924e+09,4,TRAFFIC_LIGHT,36.42,REACTION,action=NO_REACTION|time=3.04s|control={'thrott...,NaN,0,0,0
7780,1.771924e+09,4,SPEED_LIMIT,40.76,REACTION,action=REDUCE_THROTTLE|time=1.86s|control={'th...,NaN,0,0,0
7789,1.771924e+09,4,SPEED_LIMIT,40.80,SPEED_VIOLATION,Speed 40.80 km/h > limit 30.00 km/h at Locatio...,30.0,1,1,2
7790,1.771924e+09,4,SPEED_LIMIT,36.39,REACTION,action=VIOLATION_RESOLVED|time=2.30s|reaction_...,NaN,-1,0,0
8332,1.771924e+09,4,TRAFFIC_LIGHT,26.93,REACTION,action=NO_REACTION|time=3.03s|control={'thrott...,NaN,0,0,0
8536,1.771924e+09,4,SPEED_LIMIT,40.02,REACTION,action=REDUCE_THROTTLE|time=0.76s|control={'th...,NaN,0,0,0


In [125]:
df_simulation["event_start"] = (
    (df_simulation["violation_flag"] == 1) &
    (df_simulation["violation_flag"].shift(1) != 1)
).astype(int)

In [126]:
df_simulation["event_id"] = df_simulation["event_start"].cumsum()
df_simulation.loc[df_simulation["violation_flag"] != 1, "event_id"] = 0

In [127]:
display(df_simulation.head(15))

,timestamp,phase,scenario,speed_kmh,event,details,speed_limit,violation_flag,event_start,event_id
1300,1.771924e+09,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,...",30.0,0,0,0
1301,1.771924e+09,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,...",30.0,0,0,0
1302,1.771924e+09,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,...",30.0,0,0,0
1303,1.771924e+09,4,WEBSOCKET,1.17,WS_SEND,"{""phase"": 4, ""speed"": 1.17, ""speed_limit"": 30,...",30.0,0,0,0
1304,1.771924e+09,4,WEBSOCKET,1.19,WS_SEND,"{""phase"": 4, ""speed"": 1.19, ""speed_limit"": 30,...",30.0,0,0,0
1305,1.771924e+09,4,WEBSOCKET,1.19,WS_SEND,"{""phase"": 4, ""speed"": 1.19, ""speed_limit"": 30,...",30.0,0,0,0
1306,1.771924e+09,4,WEBSOCKET,1.19,WS_SEND,"{""phase"": 4, ""speed"": 1.19, ""speed_limit"": 30,...",30.0,0,0,0
1307,1.771924e+09,4,WEBSOCKET,1.19,WS_SEND,"{""phase"": 4, ""speed"": 1.19, ""speed_limit"": 30,...",30.0,0,0,0
1308,1.771924e+09,4,WEBSOCKET,1.14,WS_SEND,"{""phase"": 4, ""speed"": 1.14, ""speed_limit"": 30,...",30.0,0,0,0
1309,1.771924e+09,4,WEBSOCKET,1.14,WS_SEND,"{""phase"": 4, ""speed"": 1.14, ""speed_limit"": 30,...",30.0,0,0,0


In [108]:
#print(df_simulation['event'].unique())

### Hazard Detection counts

In [109]:
violations_df = df_simulation[df_simulation["event"].isin([
    "SPEED_VIOLATION",
    "RED_LIGHT_VIOLATION",
    "YELLOW_LIGHT_PASS",
    "Collision"
])]

In [110]:
violations_df = df_simulation[
    (df_simulation["violation_flag"] == 1) &
    (df_simulation["event"].isin([
        "SPEED_VIOLATION",
        "RED_LIGHT_VIOLATION",
        "YELLOW_LIGHT_PASS"
    ]))
]

In [111]:
display(violations_df.head(30))

,timestamp,phase,scenario,speed_kmh,event,details,speed_limit,violation_flag
2603,1.771924e+09,4,SPEED_LIMIT,41.77,SPEED_VIOLATION,Speed 41.77 km/h > limit 30.00 km/h at Locatio...,30.0,1
7789,1.771924e+09,4,SPEED_LIMIT,40.80,SPEED_VIOLATION,Speed 40.80 km/h > limit 30.00 km/h at Locatio...,30.0,1
8548,1.771924e+09,4,SPEED_LIMIT,40.07,SPEED_VIOLATION,Speed 40.07 km/h > limit 30.00 km/h at Locatio...,30.0,1
9534,1.771924e+09,1,SPEED_LIMIT,37.14,SPEED_VIOLATION,Speed 37.14 km/h > limit 30.00 km/h at Locatio...,30.0,1
9541,1.771925e+09,1,SPEED_LIMIT,48.77,SPEED_VIOLATION,Speed 48.77 km/h > limit 30.00 km/h at Locatio...,30.0,1
15948,1.771925e+09,3,SPEED_LIMIT,73.08,SPEED_VIOLATION,Speed 73.08 km/h > limit 60.00 km/h at Locatio...,60.0,1
17639,1.771925e+09,3,SPEED_LIMIT,48.73,SPEED_VIOLATION,Speed 48.73 km/h > limit 30.00 km/h at Locatio...,30.0,1
20390,1.771926e+09,2,SPEED_LIMIT,45.29,SPEED_VIOLATION,Speed 45.29 km/h > limit 30.00 km/h at Locatio...,30.0,1
20393,1.771926e+09,2,SPEED_LIMIT,98.18,SPEED_VIOLATION,Speed 98.18 km/h > limit 90.00 km/h at Locatio...,90.0,1
20399,1.771926e+09,2,SPEED_LIMIT,40.10,SPEED_VIOLATION,Speed 40.10 km/h > limit 30.00 km/h at Locatio...,30.0,1


In [112]:
violations_df["is_violation"] = violations_df["event"].isin([
    "SPEED_VIOLATION",
    "RED_LIGHT_VIOLATION",
    "YELLOW_LIGHT_PASS"
])

violations_df["violation_change"] = violations_df["is_violation"].astype(int).diff()

/var/folders/s7/6zmqmfgn45v2bblppz_50xmr0000gq/T/ipykernel_16223/1383508971.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  violations_df["is_violation"] = violations_df["event"].isin([
/var/folders/s7/6zmqmfgn45v2bblppz_50xmr0000gq/T/ipykernel_16223/1383508971.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  violations_df["violation_change"] = violations_df["is_violation"].astype(int).diff()


In [113]:
display(violations_df)

,timestamp,phase,scenario,speed_kmh,event,details,speed_limit,violation_flag,is_violation,violation_change
2603,1.771924e+09,4,SPEED_LIMIT,41.77,SPEED_VIOLATION,Speed 41.77 km/h > limit 30.00 km/h at Locatio...,30.0,1,True,NaN
7789,1.771924e+09,4,SPEED_LIMIT,40.80,SPEED_VIOLATION,Speed 40.80 km/h > limit 30.00 km/h at Locatio...,30.0,1,True,0.0
8548,1.771924e+09,4,SPEED_LIMIT,40.07,SPEED_VIOLATION,Speed 40.07 km/h > limit 30.00 km/h at Locatio...,30.0,1,True,0.0
9534,1.771924e+09,1,SPEED_LIMIT,37.14,SPEED_VIOLATION,Speed 37.14 km/h > limit 30.00 km/h at Locatio...,30.0,1,True,0.0
9541,1.771925e+09,1,SPEED_LIMIT,48.77,SPEED_VIOLATION,Speed 48.77 km/h > limit 30.00 km/h at Locatio...,30.0,1,True,0.0
...,...,...,...,...,...,...,...,...,...,...
741594,1.771681e+09,3,SPEED_LIMIT,37.26,SPEED_VIOLATION,Speed 37.26 km/h > limit 30.00 km/h at Locatio...,30.0,1,True,0.0
742398,1.771681e+09,3,SPEED_LIMIT,45.80,SPEED_VIOLATION,Speed 45.80 km/h > limit 30.00 km/h at Locatio...,30.0,1,True,0.0
744111,1.771681e+09,3,SPEED_LIMIT,48.05,SPEED_VIOLATION,Speed 48.05 km/h > limit 30.00 km/h at Locatio...,30.0,1,True,0.0
745531,1.771682e+09,3,SPEED_LIMIT,42.52,SPEED_VIOLATION,Speed 42.52 km/h > limit 30.00 km/h at Locatio...,30.0,1,True,0.0


In [114]:
df_simulation['scenario'].unique()

array(['WEBSOCKET', 'TRAFFIC_LIGHT', 'SPEED_LIMIT', 'STOP', 'COLLISION',
       'HUD'], dtype=object)

In [115]:
df_simulation['event'].unique()

array(['WS_SEND', 'YELLOW_LIGHT_PASS', 'REACTION', 'SPEED_VIOLATION',
       'RED_LIGHT_VIOLATION', 'PHASE_STOP', 'HUD_CUE'], dtype=object)

In [116]:
violations_df.to_csv("violation_Sim.csv", index=False)

In [117]:
df_simulation.to_csv("cleaned_Sim.csv", index=False)

# NASA TLX CLEANING

In [118]:
files = glob.glob("/Users/karencafe/Desktop/ThesisTest/NASACSV/*.csv")
# change to the path of the NASATLX folder containing all the csv files

df2 = pd.concat(
    (pd.read_csv(f) for f in files),
    ignore_index=True
)

display(df2)


,Participant,Task Id,Task Name,Mental Demand,Physical Demand,Temporal Demand,Performance,Effort,Frustration,Raw Score,MD Weight,PD Weight,TD Weight,PF Weight,EF Weight,FR Weight,Weighted Score
0,8,1,AH,70,60,40,70,60,80,63.333333,-1,-1,-1,-1,-1,-1,-1
1,8,2,NC,80,70,70,70,70,70,71.666667,-1,-1,-1,-1,-1,-1,-1
2,8,3,AO,80,70,70,70,70,70,71.666667,-1,-1,-1,-1,-1,-1,-1
3,8,4,HO,80,70,70,70,70,70,71.666667,-1,-1,-1,-1,-1,-1,-1
4,9,1,NC,20,40,40,60,60,40,43.333333,-1,-1,-1,-1,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,11,5,NC,5,15,5,65,50,0,23.333333,-1,-1,-1,-1,-1,-1,-1
68,5,1,AH,80,80,60,20,20,20,46.666667,-1,-1,-1,-1,-1,-1,-1
69,5,2,NC,80,80,60,10,10,10,41.666667,-1,-1,-1,-1,-1,-1,-1
70,5,3,HO,90,90,70,20,20,20,51.666667,-1,-1,-1,-1,-1,-1,-1


In [119]:
df2 = df2.drop(columns=["Task Id", "MD Weight", "PD Weight", "TD Weight", "PF Weight", "EF Weight", "FR Weight", "Weighted Score"])

display(df2)

,Participant,Task Name,Mental Demand,Physical Demand,Temporal Demand,Performance,Effort,Frustration,Raw Score
0,8,AH,70,60,40,70,60,80,63.333333
1,8,NC,80,70,70,70,70,70,71.666667
2,8,AO,80,70,70,70,70,70,71.666667
3,8,HO,80,70,70,70,70,70,71.666667
4,9,NC,20,40,40,60,60,40,43.333333
...,...,...,...,...,...,...,...,...,...
67,11,NC,5,15,5,65,50,0,23.333333
68,5,AH,80,80,60,20,20,20,46.666667
69,5,NC,80,80,60,10,10,10,41.666667
70,5,HO,90,90,70,20,20,20,51.666667


In [120]:
df2["Raw Score"] = df2["Raw Score"].round(2)
display(df2.head(3))

,Participant,Task Name,Mental Demand,Physical Demand,Temporal Demand,Performance,Effort,Frustration,Raw Score
0,8,AH,70,60,40,70,60,80,63.33
1,8,NC,80,70,70,70,70,70,71.67
2,8,AO,80,70,70,70,70,70,71.67


In [121]:
# Makes sure no mislabeled Task Name
print(df2['Task Name'].unique())

['AH' 'NC' 'AO' 'HO']


Clean task name 
- NC = no cue / baseline 
- AO = Audio Only 
- HO = Haptic Only 
- AH = Audio and Haptic / both Combined 

In [122]:
df2.columns = df2.columns.str.strip()          # removes hidden spaces
df2.columns = df2.columns.str.replace(" ", "_")

In [123]:
df2.to_csv("cleaned_NASATLX.csv", index=False)